In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sys
import os
sys.path.append(os.path.abspath(".."))
from core.viz import plot_bar, plot_dynamic_trends, plot_line, plot_corr_triangle, plot_scatter, plot_statistical_strip, plot_heatmap
from core.s3 import S3AssetManager

In [2]:

s3_cliente = S3AssetManager(notebook_name="okuo__rancimat_cliente")
s3_lab = S3AssetManager(notebook_name="okuo__rancimat_lab")

In [3]:
df = s3_cliente.read_excel(
    f"raw/rancimat/Resultado de Rancimat.xlsx",
 sheet_name="Hoja1", 
 skiprows=3
 )

cls_num = [
'INDUCCIÓN 100 °C',
'INIDUCCIÓN 120 °C', 
'DIAS', 'MESES',
'AÑOS',
'Peróxidos', 
'Anisidina',
'TVN'
]
for cl in cls_num:
    df[cl] = pd.to_numeric(df[cl], errors='coerce')

df["planta-producto"] = df["PLANTA"] + "-" + df["PRODUCTO"]

In [4]:
datos_agrupados = df.groupby(['planta-producto']).agg(
    count=('LOTE', "count"),
    hour_induction_100 =("INDUCCIÓN 100 °C", "median"),
    hour_induction_120 =('INIDUCCIÓN 120 °C', "median"),
    time_life_date =("DIAS", "median"),
    peroxidos=('Peróxidos', 'median'),
    anisidina=('Anisidina', 'median'),
    tvn=('TVN', 'median'),
).reset_index()
s3_lab.save_dataframe(datos_agrupados, "okuo__rancimat.csv")
datos_agrupados

/Users/juandavidrincon/Documents/hawking/.venv/lib/python3.13/site-packages/fsspec/registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


,planta-producto,count,hour_induction_100,hour_induction_120,time_life_date,peroxidos,anisidina,tvn
0,AMAGÁ-HCH,7,1.5050,0.640,1.250,12.30,28.98,2.94
1,AMAGÁ-HVP,6,10.4250,1.850,184.754,6.40,15.58,9.41
2,SIBATÉ-HCH,6,1.2725,0.575,3.958,5.05,16.47,7.22


In [5]:
datos_agrupados = df.groupby(['planta-producto']).agg(
    count=('LOTE', "count"),
    hour_induction_100 =("INDUCCIÓN 100 °C", "median"),
    hour_induction_120 =('INIDUCCIÓN 120 °C', "median"),
    time_life_date =("DIAS", "median"),
    peroxidos=('Peróxidos', 'median'),
    anisidina=('Anisidina', 'median'),
).reset_index()
s3_cliente.save_dataframe(datos_agrupados, "okuo__rancimat_sin_tvn.csv")
datos_agrupados

,planta-producto,count,hour_induction_100,hour_induction_120,time_life_date,peroxidos,anisidina
0,AMAGÁ-HCH,7,1.5050,0.640,1.250,12.30,28.98
1,AMAGÁ-HVP,6,10.4250,1.850,184.754,6.40,15.58
2,SIBATÉ-HCH,6,1.2725,0.575,3.958,5.05,16.47


In [14]:
plot_statistical_strip

<function core.viz.plot_statistical_strip(df: pandas.core.frame.DataFrame, x_col: str, y_col: str, category_order: Optional[List[str]] = None, color_map: Optional[Dict[str, str]] = None, show_boxplot: bool = True, show_mean_ci: bool = True, show_global_mean: bool = True, show_counts: bool = True, title: str = '', x_title: Optional[str] = None, y_title: Optional[str] = None, point_opacity: float = 0.6, point_size: int = 7, box_opacity: float = 0.25, height: int = 600, width: int = 1000, filename: Optional[str] = None) -> plotly.graph_objs._figure.Figure>

In [19]:
for col in [ 'INDUCCIÓN 100 °C', 'INIDUCCIÓN 120 °C','DIAS', 'Peróxidos', 'Anisidina','TVN']:
    if col == 'DIAS':
        df_ = df[df["DIAS"]<2000].copy()
    else:
        df_ = df.copy()

    if col == 'INIDUCCIÓN 120 °C':
        df_ = df[df['INIDUCCIÓN 120 °C']<800].copy()
    else:
        df_ = df.copy()

    f = plot_statistical_strip(df_, x_col="planta-producto", y_col=col,
    category_order=['SIBATÉ-HCH', 'AMAGÁ-HCH', 'AMAGÁ-HVP'],
    title=col, color_map={"SIBATÉ-HCH": "#1C8074", "AMAGÁ-HCH": "#666666"}, width=1000, height=400,
    point_size=10)
    print(f"{col.replace(' ', '_').lower()}_box.html")
    s3_cliente.save_plotly_html(f, f"{col.replace(' ', '_').lower()}_box.html")
    s3_lab.save_plotly_html(f, f"{col.replace(' ', '_').lower()}_box.html")
    f.show()

inducción_100_°c_box.html


iniducción_120_°c_box.html


dias_box.html


peróxidos_box.html


anisidina_box.html


tvn_box.html


In [25]:
df.columns

Index(['FECHA', 'PRODUCTO', 'PLANTA', 'LOTE', 'INDUCCIÓN 100 °C',
       'INIDUCCIÓN 120 °C', 'DIAS', 'MESES', 'AÑOS', 'Peróxidos', 'Anisidina',
       'TVN', 'planta-producto'],
      dtype='object')

In [28]:
plot_statistical_strip

<function core.viz.plot_statistical_strip(df: pandas.core.frame.DataFrame, x_col: str, y_col: str, category_order: Optional[List[str]] = None, color_map: Optional[Dict[str, str]] = None, show_boxplot: bool = True, show_mean_ci: bool = True, show_global_mean: bool = True, show_counts: bool = True, title: str = '', x_title: Optional[str] = None, y_title: Optional[str] = None, point_opacity: float = 0.6, point_size: int = 7, box_opacity: float = 0.25, height: int = 600, width: int = 1000, filename: Optional[str] = None) -> plotly.graph_objs._figure.Figure>

In [30]:
f = plot_statistical_strip(df, x_col="planta-producto", y_col='INDUCCIÓN 100 °C',
    category_order=['SIBATÉ-HCH', 'AMAGÁ-HCH', 'AMAGÁ-HVP'],
    title='INDUCCIÓN 100 °C', color_map={"SIBATÉ-HCH": "#1C8074", "AMAGÁ-HCH": "#666666"}, width=1000, height=400,
    point_size=10,
    y_title="Inducción (horas)")
print(f"inducción_100_°c_box.html")
f.show()
s3_cliente.save_plotly_html(f, "inducción_100_°c_box.html")
s3_lab.save_plotly_html(f, "inducción_100_°c_box.html")

inducción_100_°c_box.html


In [31]:
f = plot_statistical_strip(df[df['DIAS']<2000], x_col="planta-producto", y_col='DIAS',
    category_order=['SIBATÉ-HCH', 'AMAGÁ-HCH', 'AMAGÁ-HVP'],
    title="DIAS", color_map={"SIBATÉ-HCH": "#1C8074", "AMAGÁ-HCH": "#666666"}, width=1000, height=400,
    point_size=10,
    y_title="Días")
f.show()
s3_cliente.save_plotly_html(f, "dias_box.html")
s3_lab.save_plotly_html(f, "dias_box.html")

In [ ]:
f = plot_statistical_strip(df[df['INIDUCCIÓN 120 °C']<800], x_col="planta-producto", y_col='INIDUCCIÓN 120 °C',
    category_order=['SIBATÉ-HCH', 'AMAGÁ-HCH', 'AMAGÁ-HVP'],
    title="INDUCCIÓN 120 °C", color_map={"SIBATÉ-HCH": "#1C8074", "AMAGÁ-HCH": "#666666"}, width=1000, height=400,
    point_size=10,
    y_title="Inducción (horas)")
print(f"inducción_120_°c_box.html")
f.show()
s3_cliente.save_plotly_html(f, "inducción_120_°c_box.html")
s3_lab.save_plotly_html(f, "inducción_120_°c_box.html")

In [7]:
for p in df["planta-producto"].unique():
    f = plot_corr_triangle(df[df["planta-producto"]==p],
     value_cols=["INDUCCIÓN 100 °C", "INIDUCCIÓN 120 °C", "DIAS", "Peróxidos", "Anisidina", "TVN"],
      title=f"<b> Correlación  {p} </b>", 
    width=1000, height=300)
    print(f"correlacion_{p}.html")
    f.show()
    s3_lab.save_plotly_html(f, f"correlacion_{p}.html")

correlacion_SIBATÉ-HCH.html


correlacion_AMAGÁ-HVP.html


correlacion_AMAGÁ-HCH.html


In [8]:
for p in df["planta-producto"].unique():
    f = plot_corr_triangle(df[df["planta-producto"]==p],
     value_cols=["INDUCCIÓN 100 °C", "INIDUCCIÓN 120 °C", "DIAS", "Peróxidos", "Anisidina"],
      title=f"<b> Correlación  {p} </b>", 
    width=1000, height=300)
    name =f"correlacion_{p}_sin_tvn.html"
    f.show()
    s3_cliente.save_plotly_html(f, name)

In [9]:
f = plot_scatter(df, x_col='Anisidina', y_col='INDUCCIÓN 100 °C', group_col='PRODUCTO')
f.show()

In [10]:
import plotly.graph_objects as go
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
from typing import Optional

def plot_exponential_fit(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    title: str = "Análisis de Degradación",
    xaxis_title: str = "Variable X",
    yaxis_title: str = "Variable Y",
    outlier_threshold_y: Optional[float] = None,
    width: int = 1000,
    height: int = 600
) -> go.Figure:
    """
    Genera un gráfico de dispersión con ajuste de curva exponencial (Decay),
    destacando outliers y mostrando métricas de ajuste (R² y ecuación).

    Args:
        df (pd.DataFrame): DataFrame con los datos.
        x_col (str): Nombre de la columna X (ej. 'Anisidina').
        y_col (str): Nombre de la columna Y (ej. 'Induccion_100C').
        title (str): Título del gráfico.
        xaxis_title (str): Etiqueta eje X.
        yaxis_title (str): Etiqueta eje Y.
        outlier_threshold_y (float, optional): Valor límite en Y. Puntos por encima 
            se considerarán outliers y se excluirán del ajuste matemático.
        width/height (int): Dimensiones del gráfico.

    Returns:
        go.Figure: Objeto Plotly interactivo.
    """

    # --- 1. Preparación de Datos ---
    data = df[[x_col, y_col]].dropna()
    x = data[x_col].values
    y = data[y_col].values

    # Lógica de Filtrado (Outliers vs Datos Limpios)
    if outlier_threshold_y is not None:
        mask_clean = y < outlier_threshold_y
        x_clean, y_clean = x[mask_clean], y[mask_clean]
        x_out, y_out = x[~mask_clean], y[~mask_clean]
    else:
        x_clean, y_clean = x, y
        x_out, y_out = [], []

    # --- 2. Modelo Matemático (Exponencial con Asíntota) ---
    def modelo_exponencial(x_val, a, b, c):
        # y = a * e^(-b * x) + c
        return a * np.exp(-b * x_val) + c

    # Ajuste de curva (Curve Fitting)
    try:
        # Puntos iniciales sugeridos para cinéticas de degradación
        popt, _ = curve_fit(modelo_exponencial, x_clean, y_clean, p0=[5, 0.1, 0.5], maxfev=10000)
        
        # Generar línea suave para el gráfico
        x_line = np.linspace(x.min(), x.max(), 100)
        y_line = modelo_exponencial(x_line, *popt)
        
        # Calcular R²
        y_calc = modelo_exponencial(x_clean, *popt)
        r2 = r2_score(y_clean, y_calc)
        
        # Texto de la ecuación
        eq_text = f"<b>Modelo:</b> y = {popt[0]:.2f}e<sup>-{popt[1]:.2f}x</sup> + {popt[2]:.2f}<br><b>R²:</b> {r2:.3f}"
        fit_success = True
    except Exception as e:
        print(f"No se pudo ajustar el modelo: {e}")
        fit_success = False
        x_line, y_line = [], []
        eq_text = "Ajuste no convergente"

    # --- 3. Construcción del Gráfico (Plotly) ---
    fig = go.Figure()

    # Trace A: Datos "Limpios" (Usados para el modelo)
    fig.add_trace(go.Scatter(
        x=x_clean, y=y_clean,
        mode='markers',
        name='Datos Procesados',
        marker=dict(color='#5F8D8B', size=12, line=dict(width=1, color='white')),
        hovertemplate=f"{xaxis_title}: %{{x}}<br>{yaxis_title}: %{{y}}<extra></extra>"
    ))

    # Trace B: Outliers (Si existen) - Visualmente distintos
    if len(x_out) > 0:
        fig.add_trace(go.Scatter(
            x=x_out, y=y_out,
            mode='markers',
            name='Outliers (Excluidos)',
            marker=dict(color='#EF553B', symbol='x', size=10, line=dict(width=2)),
            hovertemplate=f"<b>OUTLIER</b><br>{xaxis_title}: %{{x}}<br>{yaxis_title}: %{{y}}<extra></extra>"
        ))

    # Trace C: Línea de Tendencia
    if fit_success:
        fig.add_trace(go.Scatter(
            x=x_line, y=y_line,
            mode='lines',
            name='Ajuste Exponencial',
            line=dict(color='#FF5733', width=3, dash='dash'),
            hoverinfo='skip'
        ))

    # --- 4. Layout Corporativo ---
    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor='center', font=dict(size=18)),
        width=width,
        height=height,
        template="plotly_white",
        showlegend=True,
        legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99, bgcolor="rgba(255,255,255,0.8)"),
        margin=dict(l=60, r=40, t=80, b=60),
        font=dict(family="Inter, Arial, sans-serif", color="#1F2937"),
        hovermode="closest"
    )

    # Ejes
    fig.update_xaxes(title=xaxis_title, showgrid=True, gridcolor='#E5E7EB', zeroline=False)
    fig.update_yaxes(title=yaxis_title, showgrid=True, gridcolor='#E5E7EB', zeroline=False)

    # Anotación con la Ecuación (Estilo tarjeta flotante)
    if fit_success:
        fig.add_annotation(
            x=0.98, y=0.05,
            xref="paper", yref="paper",
            text=eq_text,
            showarrow=False,
            align="right",
            bgcolor="#F3F4F6",
            bordercolor="#D1D5DB",
            borderwidth=1,
            borderpad=10,
            font=dict(size=12, color="#374151")
        )

    return fig

In [11]:
cond1 = df["PRODUCTO"] == "HCH"
cond2 = df["Anisidina"] > 10
df_hch = df[cond1 & cond2]

fig = plot_exponential_fit(
    df=df_hch,
    x_col='Anisidina',
    y_col='DIAS',
    title="<b>Tiempo vida media por Anisidina en HCH</b>",
    xaxis_title="Anisidina (Oxidación Secundaria)",
    yaxis_title="Tiempo de vida media (Días)",
    outlier_threshold_y=20  # Aquí aplicamos el filtro de "Outlier > 3.0"
)
#s3.save_plotly_html(fig, "tiempo_vida_media_por_anisidina_en_hch.html")
fig.show()

In [12]:
cond1 = df["PRODUCTO"] == "HVP"
df_hvp = df[cond1]
fig = plot_exponential_fit(
    df=df_hvp,
    x_col='Anisidina',
    y_col='MESES',
    title="<b>Tiempo vida media por Anisidina en HVP</b>",
    xaxis_title="Anisidina (Oxidación Secundaria)",
    yaxis_title="Tiempo de vida media (Meses)",
    outlier_threshold_y=500  # Aquí aplicamos el filtro de "Outlier > 3.0"
)
#s3.save_plotly_html(fig, "tiempo_vida_media_por_anisidina_en_hvp.html")

fig.show()

/var/folders/1g/77kw2x4j5678s_87_sqc1fpc0000gp/T/ipykernel_59438/3428937534.py:58: OptimizeWarning:

Covariance of the parameters could not be estimated



In [13]:


fig = plot_exponential_fit(
    df=df_hch,
    x_col='INDUCCIÓN 100 °C',
    y_col='TVN',
    title="<b>Tiempo vida media por Anisidina en HVP</b>",
    xaxis_title="Anisidina (Oxidación Secundaria)",
    yaxis_title="Tiempo de vida media (Meses)",
    outlier_threshold_y=500  # Aquí aplicamos el filtro de "Outlier > 3.0"
)

fig.show()